In [32]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rc('xtick', direction='in', top=True)
mpl.rc('ytick', direction='in', right=True)
mpl.rc('xtick.minor', visible=True)
mpl.rc('ytick.minor', visible=True)
import os
import json
rng = np.random.default_rng()



# input and output loading and creation

In [35]:
base_path1="Band_class_training data/TE_011/Results"
sidebandspath1 = os.path.join(base_path1, "Te011_sidebands.json")
with open(sidebandspath1, "r") as f:
    TE011_Sidebands_Index = json.load(f)
maxbandspath1 = os.path.join(base_path1, "Te011_MaxBands.json")
with open(maxbandspath1, "r") as f:
    TE011_Maxbands_Index = json.load(f)


In [36]:
base_path2="Band_class_training data/TE_012/Results"
sidebandspath2 = os.path.join(base_path2, "Te012_sidebands.json")
with open(sidebandspath2, "r") as f:
    TE012_Sidebands_Index = json.load(f)
maxbandspath2 = os.path.join(base_path2, "Te012_MaxBands.json")
with open(maxbandspath2, "r") as f:
    TE012_Maxbands_Index = json.load(f)


In [37]:
def make_inputs(index1,index2,numpeaks1=5,numpeaks2=5,numfeatures=2):
    first_angle = next(iter(index1))
    first_position = next(iter(index1[first_angle]))

    Npsd=len(index1)*len(index1[first_angle])*(len(index1[first_angle][first_position]))
    
    empt=np.zeros((Npsd,numpeaks1+numpeaks2,numfeatures))
    jsn_name = "LocustEventProperties.json"
    band_name='side_bands_powercenter'
    i=0
    for angle in index1:
        for position in index1[angle]:
            for hist in index1[angle][position]:
                 if hist != jsn_name:
                     peaks1=np.array(index1[angle][position][hist][band_name])
                     peaks2=np.array(index2[angle][position][hist][band_name])

                     if peaks1.shape[0] != numpeaks1 or peaks2.shape[0] != numpeaks2:
                        raise ValueError("Expected 5 peaks per index")

                     empt[i, :, :] = np.vstack([peaks1, peaks2])
                    
                     i=i+1
    full=empt
    return(full)

In [38]:
def make_outputs(max_array,band_array,freq_samp=50000,binspace=4):
    empt = np.zeros(max_array.shape[:2])
    for i in range(len(max_array)):
        ar1=max_array[i]
        ar2=band_array[i]
        spread=freq_samp*binspace
        match_matrix = (np.abs(ar1[:, None, :] - ar2[None, :, :2]) <= spread).all(axis=2)
        ar1_i, ar2_i = np.where(match_matrix)
        out = np.full(ar1.shape[0], np.inf)
        out[ar1_i] = ar2[ar2_i, 2]
        empt[i]=out
    return(empt)

    


In [39]:
inputs=make_inputs(TE011_Maxbands_Index,TE012_Maxbands_Index)


In [40]:
band1=make_inputs(TE011_Sidebands_Index,TE012_Sidebands_Index,numpeaks1=3,numpeaks2=2,numfeatures=3)


In [41]:
outputs=make_outputs(inputs,band1)

In [42]:
def split_random_rows(arr1, arr2, percent,rng=None):
    if arr1.shape[0] != arr2.shape[0]:
        raise ValueError("arr1 and arr2 must have the same number of rows.")

    if not (0 <= percent <= 100):
        raise ValueError("percent must be between 0 and 100.")

    if rng is None:
        rng = np.random.default_rng()

    N = arr1.shape[0]
    n_remove = int(round(N * percent / 100))

    removed_indices = rng.choice(N, size=n_remove, replace=False)

    # Boolean mask for remaining rows
    mask = np.ones(N, dtype=bool)
    mask[removed_indices] = False

    arr1_remaining = arr1[mask]
    arr2_remaining = arr2[mask]

    arr1_removed = arr1[removed_indices]
    arr2_removed = arr2[removed_indices]

    return (
        arr1_remaining,
        arr2_remaining,
        arr1_removed,
        arr2_removed,
        removed_indices,
    )

In [43]:
model_inputs,model_outputs,testing_inputs,testing_outputs,indx=split_random_rows(inputs,outputs,1)

# Py torch prep

In [44]:
import torch
import sklearn
import tqdm
import tensorboard

In [69]:
label_to_class = {
    -2: 0,
    -1: 1,
     0: 2,
     1: 3,
     2: 4,
}

class_to_label = {
    0: -2,
    1: -1,
    2: 0,
    3: 1,
    4: 2,
    5: np.inf,
}

def band_encoding(outputs):
    outputs_incoded =np.empty(outputs.shape,dtype=np.int64)

    outputs_incoded[outputs == -2] = 0
    outputs_incoded[outputs == -1] = 1
    outputs_incoded[outputs ==  0] = 2
    outputs_incoded[outputs ==  1] = 3
    outputs_incoded[outputs ==  2] = 4
    outputs_incoded[np.isinf(outputs)] = 5

    return outputs_incoded
incoded_outputs=band_encoding(model_outputs)

In [46]:
#rescaling dataa

X_train, X_val, y_train, y_val = sklearn.model_selection.train_test_split(
    model_inputs,
    incoded_outputs,
    test_size=0.2,
    random_state=0,
    shuffle=True
)

scaler=sklearn.preprocessing.StandardScaler()

X_train_2d = X_train.reshape(-1, 2)
scaler.fit(X_train_2d)

X_train_scaled = scaler.transform(X_train.reshape(-1, 2)).reshape(X_train.shape)
X_val_scaled = scaler.transform(X_val.reshape(-1, 2)).reshape(X_val.shape)

In [47]:
from torch.utils.data import TensorDataset, DataLoader

X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)

X_val_t = torch.tensor(X_val_scaled, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.long)

train_ds = TensorDataset(X_train_t, y_train_t)
val_ds = TensorDataset(X_val_t, y_val_t)

train_loader = DataLoader(
    train_ds,
    batch_size=128,
    shuffle=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=256,
    shuffle=False
)

# model prep and


In [76]:
#definign model
import torch.nn as nn


class PeakTransformer(nn.Module):
    def __init__(
        self,
        n_features=2,
        n_classes=6,
        d_model=64,
        n_heads=4,
        n_layers=2,
        dim_feedforward=128,
        dropout=0.1
    ):
        super().__init__()

        self.input_proj = nn.Linear(n_features, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=n_layers
        )

        self.classifier = nn.Linear(d_model, n_classes)

    def forward(self, x):
        """
        x shape: (batch_size, 10, 2)

        returns:
            logits shape: (batch_size, 10, 6)
        """

        x = self.input_proj(x)
        x = self.encoder(x)
        logits = self.classifier(x)

        return logits

In [77]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = PeakTransformer(
    n_features=2,
    n_classes=6,
    d_model=64,
    n_heads=4,
    n_layers=2
).to(device)

counts = torch.bincount(y_train_t.reshape(-1), minlength=6).float()

# inverse-frequency style weights
weights = counts.sum() / (6 * counts)

# avoid problems if one class is missing from training
weights[torch.isinf(weights)] = 0.0
weights = weights.to(device)

print("Class counts:", counts)
print("Class weights:", weights)

Class counts: tensor([ 20811.,  21980.,  22249.,  22044.,   4040., 133396.])
Class weights: tensor([1.7981, 1.7025, 1.6819, 1.6975, 9.2624, 0.2805])


# Training DO NOT RUN

In [78]:
#trainging DO NOT RUN

criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

n_epochs = 30

for epoch in range(n_epochs):
    model.train()
    train_loss = 0.0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        logits = model(X_batch)

        # logits shape: (batch, 10, 6)
        # y_batch shape: (batch, 10)
        #
        # flatten so CrossEntropyLoss sees:
        # logits: (batch * 10, 6)
        # labels: (batch * 10)
        loss = criterion(
            logits.reshape(-1, 6),
            y_batch.reshape(-1)
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * X_batch.size(0)

    train_loss /= len(train_loader.dataset)

    # validation
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            logits = model(X_batch)

            loss = criterion(
                logits.reshape(-1, 6),
                y_batch.reshape(-1)
            )

            val_loss += loss.item() * X_batch.size(0)

            predictions = torch.argmax(logits, dim=-1)

            correct += (predictions == y_batch).sum().item()
            total += y_batch.numel()

    val_loss /= len(val_loader.dataset)
    val_acc = correct / total

    print(
        f"Epoch {epoch + 1:03d} | "
        f"train loss = {train_loss:.4f} | "
        f"val loss = {val_loss:.4f} | "
        f"val acc = {val_acc:.4f}"
    )

Epoch 001 | train loss = 0.3349 | val loss = 0.0547 | val acc = 0.9734
Epoch 002 | train loss = 0.0678 | val loss = 0.0262 | val acc = 0.9827
Epoch 003 | train loss = 0.0457 | val loss = 0.0161 | val acc = 0.9880
Epoch 004 | train loss = 0.0372 | val loss = 0.0148 | val acc = 0.9872
Epoch 005 | train loss = 0.0332 | val loss = 0.0157 | val acc = 0.9873
Epoch 006 | train loss = 0.0308 | val loss = 0.0117 | val acc = 0.9904
Epoch 007 | train loss = 0.0279 | val loss = 0.0142 | val acc = 0.9940
Epoch 008 | train loss = 0.0265 | val loss = 0.0138 | val acc = 0.9919
Epoch 009 | train loss = 0.0232 | val loss = 0.0101 | val acc = 0.9926
Epoch 010 | train loss = 0.0223 | val loss = 0.0072 | val acc = 0.9943
Epoch 011 | train loss = 0.0231 | val loss = 0.0099 | val acc = 0.9924
Epoch 012 | train loss = 0.0202 | val loss = 0.0077 | val acc = 0.9929
Epoch 013 | train loss = 0.0202 | val loss = 0.0071 | val acc = 0.9932
Epoch 014 | train loss = 0.0206 | val loss = 0.0079 | val acc = 0.9932
Epoch 

# Testing

In [79]:
def classify_one_psd(model, single_psd, scaler, device):
    """
    single_psd shape: (10, 2)

    Returns:
        predicted_classes : (10,) integer classes (0-5)
        confidences       : (10,) probability of the predicted class
    """

    model.eval()

    # Scale features
    single_scaled = scaler.transform(single_psd)

    # Add batch dimension
    X = torch.tensor(
        single_scaled,
        dtype=torch.float32
    ).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(X)
        probabilities = torch.softmax(logits, dim=-1)

    # Remove batch dimension
    probabilities = probabilities.squeeze(0)

    # Most likely class for each peak
    predicted_classes = torch.argmax(probabilities, dim=1)

    # Probability of the predicted class
    confidences = probabilities.gather(
        1,
        predicted_classes.unsqueeze(1)
    ).squeeze(1)

    predicted_classes=predicted_classes.cpu().numpy()

    predicted_labels = np.array([
        class_to_label[c] for c in predicted_classes
    ])

    return predicted_labels, confidences.cpu().numpy()


In [87]:
def model_acuracy(testing_inputs,testing_outputs,model,scaler,device):
    empt=np.zeros(testing_outputs.shape[0])

    for i in range(len(empt)):
        prediction,confidence=classify_one_psd(model,testing_inputs[i],scaler,device)
        sim=np.mean(prediction==testing_outputs[i])
        empt[i]=sim
        print(confidence)
    acc=np.mean(empt)
    return(acc)

In [88]:
acc=model_acuracy(testing_inputs,testing_outputs,model,scaler,device)
print(acc)


[0.9999069  0.99999154 0.9999982  0.99999344 0.9999945  0.99998474
 0.9999987  0.9998431  0.9999995  0.99978393]
[0.9999815  0.9998074  0.99999833 0.9998466  0.99997246 0.99995553
 0.99999976 0.99980885 0.99999976 0.99974567]
[0.9999808  0.99989927 0.99998665 0.9998344  0.9999995  0.99997985
 0.99993026 0.99999976 0.9999989  0.99981564]
[0.99997854 0.9999764  0.99993396 0.9999993  0.9999535  0.9999783
 0.99963367 0.9995548  0.99999845 0.9985812 ]
[0.9999807  0.9999666  0.99993086 0.9999995  0.9999713  0.9999546
 0.99999976 0.9998989  0.9999989  0.9998697 ]
[0.9988342  0.9999801  0.9999963  0.99995136 0.99998724 0.99986076
 0.99941635 0.99999857 0.9998882  0.9999993 ]
[0.9999809 0.9999614 0.9999883 0.9999473 0.9999994 0.9999795 0.9999244
 0.9999995 0.9997124 0.9999976]
[0.999982   0.9985343  0.99992454 0.9999833  0.9999647  0.99997854
 0.9999807  0.9999913  0.999998   0.9998166 ]
[0.99956757 0.9996619  0.9999908  0.9990588  0.9999503  0.99707675
 0.99999297 0.9979278  0.99999905 0.98914

In [82]:
testing_inputs[1]

array([[3.61081448e-12, 1.23330200e+08],
       [2.24607971e-13, 1.37891724e+08],
       [2.32049385e-11, 1.38137695e+08],
       [2.92106564e-13, 1.38383667e+08],
       [2.39083225e-14, 1.38629639e+08],
       [2.64636044e-12, 1.15852661e+08],
       [7.21812389e-12, 1.30660156e+08],
       [6.46843769e-14, 1.45221680e+08],
       [1.83990211e-11, 1.45467651e+08],
       [9.39243646e-14, 1.45713623e+08]])

In [83]:
prediction,confidence=classify_one_psd(model,testing_inputs[1],scaler,device)


In [84]:
print(prediction)
print(testing_outputs[1])

[-2. inf  0. inf inf inf -1. inf  1. inf]
[-2. inf  0. inf inf inf -1. inf  1. inf]
